# Задание 2. Эмпирический закон Ципфа

Проверяется утверждение: *длина слова обратно пропорциональна его частоте* (для служебных слов).

В качестве списка служебных слов использовались союзы, предлоги, частицы, местоимения и вспомогательные глаголы. Для каждого такого слова определялись его частота в корпусе, длина в буквах, относительная частота (на 10^6 токенов).

Анализ выполнялся для всех четырёх словарей. Рассчитывались коэффициенты корреляции Пирсона и Спирмена между длиной слова и его частотой (а также между длиной слова и $\log_{10} f$), строилась регрессия $\log_{10} f \approx a + b \cdot L$, где $L$ — длина слова.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'freq_ru.json').exists():
    ROOT = ROOT.parent
BASE = ROOT / 'rgr2'
DATA = BASE / 'data'
FIG = BASE / 'figures'
DATA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 220,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

FUNCTION_WORDS_RU = [
    "в", "во", "на", "с", "со", "по", "к", "ко", "у", "за", "из", "от", "до", "для", "без", "под", "над", "при", "про",
    "через", "между", "перед", "около", "после", "кроме", "ради", "благодаря",
    "и", "а", "но", "да", "или", "либо", "если", "что", "чтобы", "когда", "пока", "потому", "так", "как", "тоже", "также",
    "ни", "то", "не", "же", "ли", "бы", "ведь", "вот", "вон", "даже", "уже", "еще", "только", "именно", "почти",
    "просто", "разве", "неужели",
    "я", "ты", "он", "она", "оно", "мы", "вы", "они", "меня", "тебя", "его", "ее", "её", "нас", "вас", "их", "мне", "тебе",
    "ему", "ей", "нам", "вам", "им", "себя", "себе", "собой", "себя",
    "этот", "эта", "это", "эти", "тот", "та", "то", "те", "такой", "такая", "такое", "такие", "весь", "вся", "все",
    "свой", "мой", "моя", "моё", "мое", "мои", "твой", "твоё", "твое", "твои", "наш", "наша", "наше", "наши", "ваш",
    "ваша", "ваше", "ваши",
    "кто", "что", "какой", "какая", "какое", "какие", "который", "которая", "которое", "которые", "чей", "чья", "чьё", "чье", "чьи",
    "где", "куда", "откуда", "когда", "почему", "зачем", "сколько",
    "быть", "был", "была", "было", "были", "будет", "буду", "будешь", "будем", "будете", "будут", "есть", "бытие"
]

FUNCTION_WORDS_EN = [
    "the", "a", "an",
    "of", "in", "to", "for", "with", "on", "at", "by", "from", "up", "about", "into", "over", "after", "beneath", "under",
    "above", "across", "along", "among", "around", "before", "behind", "below", "beside", "between", "beyond", "during",
    "except", "inside", "outside", "since", "through", "toward", "towards", "upon", "within", "without",
    "and", "or", "but", "nor", "so", "yet", "because", "although", "though", "if", "unless", "until", "while", "whereas",
    "whether", "as", "than", "that", "this", "these", "those",
    "i", "you", "he", "she", "it", "we", "they", "me", "him", "her", "us", "them", "my", "your", "his", "its", "our", "their",
    "mine", "yours", "hers", "ours", "theirs",
    "myself", "yourself", "himself", "herself", "itself", "ourselves", "yourselves", "themselves",
    "who", "whom", "whose", "which", "what", "where", "when", "why", "how",
    "be", "am", "is", "are", "was", "were", "been", "being",
    "have", "has", "had", "having",
    "do", "does", "did", "doing", "done",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must", "need", "dare", "ought",
    "not", "no", "yes", "too", "very", "just", "only", "also", "even", "well", "there", "here", "now", "then", "thus",
    "hence", "however", "therefore", "already", "still", "again", "ever", "never", "always", "often", "sometimes",
    "once", "soon", "almost", "quite", "rather", "perhaps", "maybe"
]

def dedup_preserve(seq):
    seen = set()
    out = []
    for x in seq:
        if x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out

FUNCTION_WORDS_RU = dedup_preserve(FUNCTION_WORDS_RU)
FUNCTION_WORDS_EN = dedup_preserve(FUNCTION_WORDS_EN)

DICTIONARIES = [
    ('ru_raw',   'Русский, без лемматизации',  'freq_ru.json',   FUNCTION_WORDS_RU),
    ('ru_lemma', 'Русский, с лемматизацией',   'freq_ru_lemma.json', FUNCTION_WORDS_RU),
    ('en_raw',   'Английский, без лемматизации', 'freq_en.json',   FUNCTION_WORDS_EN),
    ('en_lemma', 'Английский, с лемматизацией',  'freq_en_lemma.json', FUNCTION_WORDS_EN),
]

def load_dictionary(filename):
    with open(ROOT / filename, encoding='utf-8') as f:
        package = json.load(f)
    return package, package['dictionary']

packages = {}
for key, label, filename, fw in DICTIONARIES:
    package, dct = load_dictionary(filename)
    packages[key] = {'package': package, 'dict': dct, 'label': label, 'fw': fw}
    print(f"{label}: токенов {package['total_tokens']:,}")


Русский, без лемматизации: токенов 4,982,552
Русский, с лемматизацией: токенов 5,000,251


Английский, без лемматизации: токенов 5,002,249
Английский, с лемматизацией: токенов 5,002,249


In [2]:
rows = []
fw_dfs = {}

for key, label, filename, fw in DICTIONARIES:
    package = packages[key]['package']
    dct = packages[key]['dict']
    total = package['total_tokens']
    data = []
    for w in fw:
        if w in dct:
            f = dct[w]
            data.append({
                'word': w,
                'freq': int(f),
                'length': len(w),
                'rel_freq_per_1e6': f / total * 1e6,
                'log10_freq': np.log10(f) if f > 0 else np.nan,
            })
    df = pd.DataFrame(data)
    df['rank_in_dict'] = df['freq'].rank(ascending=False, method='min').astype(int)
    fw_dfs[key] = df
    df.to_csv(DATA / f'function_words_{key}.csv', index=False, float_format='%.6f')

    if len(df) >= 3:
        pear_freq = stats.pearsonr(df['length'], df['freq'])
        pear_log = stats.pearsonr(df['length'], df['log10_freq'])
        spear_freq = stats.spearmanr(df['length'], df['freq'])
        if len(df) >= 2:
            slope, intercept, r, p, se = stats.linregress(df['length'], df['log10_freq'])
            r2 = r * r
        else:
            slope, intercept, r2, p = np.nan, np.nan, np.nan, np.nan

        rows.append({
            'key': key,
            'dictionary': label,
            'n_function_words': int(len(df)),
            'pearson_length_freq': float(pear_freq.statistic),
            'p_pearson_freq': float(pear_freq.pvalue),
            'pearson_length_log10f': float(pear_log.statistic),
            'p_pearson_log10f': float(pear_log.pvalue),
            'spearman_length_freq': float(spear_freq.statistic),
            'p_spearman_freq': float(spear_freq.pvalue),
            'regress_log10f_b': float(slope),
            'regress_log10f_a': float(intercept),
            'r2_log10f': float(r2),
        })
    else:
        rows.append({
            'key': key, 'dictionary': label, 'n_function_words': int(len(df)),
            'pearson_length_freq': np.nan, 'p_pearson_freq': np.nan,
            'pearson_length_log10f': np.nan, 'p_pearson_log10f': np.nan,
            'spearman_length_freq': np.nan, 'p_spearman_freq': np.nan,
            'regress_log10f_b': np.nan, 'regress_log10f_a': np.nan, 'r2_log10f': np.nan,
        })

task2_summary = pd.DataFrame(rows)
task2_summary.to_csv(DATA / 'zipf2_function_words_summary.csv', index=False, float_format='%.8f')
display(task2_summary.rename(columns={
    'dictionary': 'Словарь', 'n_function_words': 'Служ. слов',
    'pearson_length_freq': 'r(Pearson, L,f)', 'pearson_length_log10f': 'r(Pearson, L,log10 f)',
    'spearman_length_freq': 'ρ(Spearman, L,f)',
    'regress_log10f_b': 'b (наклон)', 'r2_log10f': 'R²',
}).round(3))


,key,Словарь,Служ. слов,"r(Pearson, L,f)",p_pearson_freq,"r(Pearson, L,log10 f)",p_pearson_log10f,"ρ(Spearman, L,f)",p_spearman_freq,b (наклон),regress_log10f_a,R²
0,ru_raw,"Русский, без лемматизации",151,-0.311,0.000,-0.297,0.0,-0.319,0.0,-0.207,3.620,0.088
1,ru_lemma,"Русский, с лемматизацией",93,-0.330,0.001,-0.381,0.0,-0.400,0.0,-0.227,4.137,0.145
2,en_raw,"Английский, без лемматизации",164,-0.287,0.000,-0.485,0.0,-0.479,0.0,-0.246,4.367,0.235
3,en_lemma,"Английский, с лемматизацией",145,-0.331,0.000,-0.492,0.0,-0.488,0.0,-0.264,4.439,0.242


## Графики для задания 2

Слева — зависимость относительной частоты служебных слов от их длины. Справа — зависимость $\log_{10} f$ от длины слова с линией линейной регрессии.


In [3]:
def make_task2_figure(keys, title, out_path):
    fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.0))
    for col, key in enumerate(keys):
        df = fw_dfs[key]
        label = packages[key]['label']
        row = task2_summary.set_index('key').loc[key]

        ax = axes[0, col]
        ax.scatter(df['length'], df['rel_freq_per_1e6'], s=30, alpha=0.7, color='#1f77b4')
        for _, r in df.nlargest(12, 'freq').iterrows():
            ax.annotate(r['word'], (r['length'], r['rel_freq_per_1e6']),
                        textcoords='offset points', xytext=(3, 2), fontsize=7)
        ax.set_title(label)
        ax.set_xlabel('Длина слова L, символов')
        ax.set_ylabel('Относительная частота, на 10^6 токенов')
        ax.grid(True, alpha=0.25)

        ax = axes[1, col]
        ax.scatter(df['length'], df['log10_freq'], s=30, alpha=0.7, color='#d62728')
        if len(df) >= 2:
            x = df['length'].to_numpy(dtype=float)
            y = df['log10_freq'].to_numpy(dtype=float)
            slope = row['regress_log10f_b']
            intercept = row['regress_log10f_a']
            x_grid = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_grid, intercept + slope * x_grid, color='black', linestyle='--',
                    linewidth=1.3, label=f'log10 f = {intercept:.2f} + {slope:.3f}·L, R²={row["r2_log10f"]:.2f}')
            ax.legend(loc='best')
        for _, r in df.nlargest(10, 'freq').iterrows():
            ax.annotate(r['word'], (r['length'], r['log10_freq']),
                        textcoords='offset points', xytext=(3, 2), fontsize=7)
        ax.set_title(f'{label}: log10 f от длины')
        ax.set_xlabel('Длина слова L, символов')
        ax.set_ylabel('$\log_{10} f$')
        ax.grid(True, alpha=0.25)

    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print('Сохранено:', out_path)


make_task2_figure(['ru_raw', 'ru_lemma'], 'Задание 2. Служебные слова: частота и длина слова (русский язык)',
                  FIG / 'fig_zipf2_ru.png')
make_task2_figure(['en_raw', 'en_lemma'], 'Задание 2. Служебные слова: частота и длина слова (английский язык)',
                  FIG / 'fig_zipf2_en.png')


<>:34: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:34: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
/tmp/ipykernel_56517/3394969539.py:34: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
  ax.set_ylabel('$\log_{10} f$')


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf2_ru.png


Сохранено: /home/dd/nlp_uni/rgr2/figures/fig_zipf2_en.png


In [4]:
lines = ['### Вывод по заданию 2', '']
for _, row in task2_summary.iterrows():
    s = f"- **{row['dictionary']}** (n = {int(row['n_function_words'])} служебных слов): "
    s += f"корреляция между длиной слова и $\\log_{{10}} f$ (Пирсон) — $r = {row['pearson_length_log10f']:.3f}$, "
    s += f"между длиной и абсолютной частотой — $r = {row['pearson_length_freq']:.3f}$, "
    s += f"$\\rho_{{Спирмена}} = {row['spearman_length_freq']:.3f}$. "
    if not pd.isna(row['regress_log10f_b']):
        s += f"Регрессия: $\\log_{{10}} f = {row['regress_log10f_a']:.2f} + ({row['regress_log10f_b']:.3f})\\cdot L$, $R^2 = {row['r2_log10f']:.3f}$."
    lines.append(s)

lines += [
    '',
    'Полученные коэффициенты корреляции отрицательные, что соответствует общей тенденции: более короткие служебные слова встречаются заметно чаще длинных. '
    'Для английского языка при наличии лемматизации соотношение проявляется несколько отчётливее. '
    'Вместе с тем связь умеренная, поэтому утверждение «длина слова обратно пропорциональна его частоте» для служебных слов можно считать выполненным в качественном смысле, но не строго.'
]
display(Markdown('\n'.join(lines)))


### Вывод по заданию 2

- **Русский, без лемматизации** (n = 151 служебных слов): корреляция между длиной слова и $\log_{10} f$ (Пирсон) — $r = -0.297$, между длиной и абсолютной частотой — $r = -0.311$, $\rho_{Спирмена} = -0.319$. Регрессия: $\log_{10} f = 3.62 + (-0.207)\cdot L$, $R^2 = 0.088$.
- **Русский, с лемматизацией** (n = 93 служебных слов): корреляция между длиной слова и $\log_{10} f$ (Пирсон) — $r = -0.381$, между длиной и абсолютной частотой — $r = -0.330$, $\rho_{Спирмена} = -0.400$. Регрессия: $\log_{10} f = 4.14 + (-0.227)\cdot L$, $R^2 = 0.145$.
- **Английский, без лемматизации** (n = 164 служебных слов): корреляция между длиной слова и $\log_{10} f$ (Пирсон) — $r = -0.485$, между длиной и абсолютной частотой — $r = -0.287$, $\rho_{Спирмена} = -0.479$. Регрессия: $\log_{10} f = 4.37 + (-0.246)\cdot L$, $R^2 = 0.235$.
- **Английский, с лемматизацией** (n = 145 служебных слов): корреляция между длиной слова и $\log_{10} f$ (Пирсон) — $r = -0.492$, между длиной и абсолютной частотой — $r = -0.331$, $\rho_{Спирмена} = -0.488$. Регрессия: $\log_{10} f = 4.44 + (-0.264)\cdot L$, $R^2 = 0.242$.

Полученные коэффициенты корреляции отрицательные, что соответствует общей тенденции: более короткие служебные слова встречаются заметно чаще длинных. Для английского языка при наличии лемматизации соотношение проявляется несколько отчётливее. Вместе с тем связь умеренная, поэтому утверждение «длина слова обратно пропорциональна его частоте» для служебных слов можно считать выполненным в качественном смысле, но не строго.